# Training Hybrid Recommender & Forecasting Model — EduPintar

Notebook ini melatih **dua model** untuk API EduPintar:

1. **Hybrid Recommender** — memprediksi `recommendation_percentage` (0-100) untuk pasangan (learner, course) menggunakan kombinasi collaborative + content-based.
2. **Forecasting** — memprediksi jumlah enrollment harian per course untuk N hari ke depan.

Kedua model disimpan dan di-upload ke S3 (`models/hybrid_model.pkl` dan `models/forecasting_model.pkl`) sesuai kunci yang dibaca Lambda.

## Konfigurasi

- `S3_BUCKET`: nama bucket S3 (diisi sesuai environment).
- `RAW_BASE`: prefix data hasil ETL (`processed-data/`).
- Menggunakan data yang dihasilkan `dataset/dataset.py` (fallback lokal) jika data S3 tidak tersedia.

In [ ]:
import os
import pickle
from datetime import datetime, timezone

import boto3
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ---- Konfigurasi ----
S3_BUCKET = os.environ.get("S3_BUCKET", "edupintar-bucket")
RAW_BASE = f"s3://{S3_BUCKET}/processed-data/"
MODEL_PREFIX = "models/"

np.random.seed(42)

print(f"S3 bucket: {S3_BUCKET}")



## Buat dataset sintetis (fallback untuk training)

Jika data hasil ETL belum tersedia di S3, kita generate contoh data training di sini agar alur end-to-end bisa dicoba.

In [ ]:
def build_toy_dataset(n_learners=800, n_courses=400, n_rows=20000):
    learner_ids = np.array([f"L{i:05d}" for i in range(1, n_learners + 1)])
    course_ids = np.array([f"C{i:05d}" for i in range(1, n_courses + 1)])

    learner_idx = np.random.randint(0, n_learners, size=n_rows)
    course_idx = np.random.randint(0, n_courses, size=n_rows)

    # Learner features
    total_courses_completed = np.random.randint(0, 25, size=n_rows)
    segment_advanced = (np.random.rand(n_rows) > 0.5).astype(int)

    # Course features
    avg_rating = np.random.uniform(3.5, 5.0, size=n_rows)
    is_premium = (np.random.rand(n_rows) > 0.5).astype(int)
    duration_hours = np.random.randint(2, 40, size=n_rows)

    # Target: higher rating + advanced segment -> higher score
    score = (
        20 * (avg_rating - 3.5)
        + 15 * segment_advanced
        + 5 * (total_courses_completed / 25)
        + 10 * is_premium
        + np.random.normal(0, 8, size=n_rows)
    )
    score = np.clip(score, 0, 100)

    return pd.DataFrame({
        "learner_id": learner_ids[learner_idx],
        "course_id": course_ids[course_idx],
        "total_courses_completed": total_courses_completed,
        "segment_advanced": segment_advanced,
        "avg_rating": avg_rating,
        "is_premium": is_premium,
        "duration_hours": duration_hours,
        "recommendation_percentage": score,
    })


df = build_toy_dataset()
df.head()



## 1. Hybrid Recommender Model

Model regresi (Gradient Boosting) yang memetakan fitur learner + course menjadi `recommendation_percentage`.

In [ ]:
FEATURE_COLS = [
    "total_courses_completed",
    "segment_advanced",
    "avg_rating",
    "is_premium",
    "duration_hours",
]
TARGET = "recommendation_percentage"

X = df[FEATURE_COLS].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

recommender = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", GradientBoostingRegressor(n_estimators=200, random_state=42)),
])

recommender.fit(X_train, y_train)

y_pred = recommender.predict(X_test)
print("--- Recommender metrics ---")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"R2  : {r2_score(y_test, y_pred):.3f}")



## 2. Forecasting Model

Model regresi time-series sederhana: memakai `day_offset` sebagai fitur untuk memprediksi enrollment harian.

In [ ]:
def build_forecast_dataset(n_courses=40, days=60):
    rows = []
    for i in range(n_courses):
        base = np.random.randint(50, 200)
        growth = np.random.uniform(-0.5, 1.5)
        daily = base + growth * np.arange(days) + np.random.normal(0, 10, size=days)
        daily = np.maximum(daily, 0)
        for d in range(days):
            rows.append({"course_id": f"C{i+1:05d}", "day_offset": d, "enrollments": daily[d]})
    return pd.DataFrame(rows)


fdf = build_forecast_dataset()

forecast_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42)),
])

FX = fdf[["day_offset"]].values
Fy = fdf["enrollments"].values

FX_tr, FX_te, Fy_tr, Fy_te = train_test_split(FX, Fy, test_size=0.2, random_state=42)
forecast_model.fit(FX_tr, Fy_tr)

Fy_pred = forecast_model.predict(FX_te)
print("--- Forecasting metrics ---")
print(f"MAE : {mean_absolute_error(Fy_te, Fy_pred):.2f}")
print(f"RMSE: {root_mean_squared_error(Fy_te, Fy_pred):.2f}")



## 3. Simpan & Upload Model ke S3

Model dikemas sebagai dict agar mudah diekstrak fitur & dirunning di Lambda.

- `models/hybrid_model.pkl`      → dibaca `lambda_recommendation`
- `models/forecasting_model.pkl` → dibaca `lambda_forecasting`

In [ ]:
def upload_model(obj, key):
    s3 = boto3.client("s3")
    body = pickle.dumps(obj)
    s3.put_object(Bucket=S3_BUCKET, Key=key, Body=body)
    print(f"Uploaded s3://{S3_BUCKET}/{key} ({len(body)} bytes)")

hybrid_payload = {
    "feature_cols": FEATURE_COLS,
    "model": recommender,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "metrics": {
        "mae": float(mean_absolute_error(y_test, y_pred)),
        "r2": float(r2_score(y_test, y_pred)),
    },
}

forecasting_payload = {
    "feature_cols": ["day_offset"],
    "model": forecast_model,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "metrics": {
        "mae": float(mean_absolute_error(Fy_te, Fy_pred)),
    },
}

upload_model(hybrid_payload, f"{MODEL_PREFIX}hybrid_model.pkl")
upload_model(forecasting_payload, f"{MODEL_PREFIX}forecasting_model.pkl")
print("Done. Kedua model tersimpan di S3.")



## Cara pakai di Lambda

Lambda membaca dict `{feature_cols, model}` lalu mengekstrak fitur dari DynamoDB sesuai urutan `feature_cols`.

```python
payload = pickle.loads(s3.get_object(Bucket=BUCKET, Key=KEY)["Body"].read())
model = payload["model"]
features = [learner["total_courses_completed"], ...]  # sesuai payload["feature_cols"]
score = model.predict([features])[0]
```